In [38]:
import duckdb
import matplotlib.pyplot as plt

con = duckdb.connect("../blood_data.duckdb")


In [11]:
df = con.execute("select * from retention").df()

df

,donor_id,visit_date,birth_date
0,00000,2013-11-03,1964
1,00000,2014-05-26,1964
2,00000,2015-01-25,1964
3,00000,2015-08-09,1964
4,00000,2016-03-10,1964
...,...,...,...
6910823,00neq,2025-11-24,1993
6910824,09RaK,2025-11-24,1987
6910825,03wlA,2025-11-24,1989
6910826,03cUi,2025-11-24,1984


In [41]:
query = """
        select *, 
        lag(visit_date,1,visit_date) over(w) as previous_visit,
        lead(visit_date,1,visit_date) over(w) as next_visit,
        coalesce(date_diff('day',lag(visit_date) over w,visit_date)) as days_diff,
        case
            when lag(visit_date) over w is null then 'First Visit'
            when lead(visit_date) over w is null then 'Last Visit'
            else 'Returning'
        end as visit_status,
        row_number() over (partition by donor_id order by visit_date) as nth_visit,
        year(visit_date) - birth_date as age,

        from retention
        window w as (partition by donor_id order by visit_date)
        
        order by donor_id,visit_date
        limit 11
        """

df = con.execute(query).df()

df

,donor_id,visit_date,birth_date,previous_visit,next_visit,days_diff,visit_status,nth_visit,age
0,00000,2013-11-03,1964,2013-11-03,2014-05-26,<NA>,First Visit,1,49
1,00000,2014-05-26,1964,2013-11-03,2015-01-25,204,Returning,2,50
2,00000,2015-01-25,1964,2014-05-26,2015-08-09,244,Returning,3,51
3,00000,2015-08-09,1964,2015-01-25,2016-03-10,196,Returning,4,51
4,00000,2016-03-10,1964,2015-08-09,2016-08-14,214,Returning,5,52
5,00000,2016-08-14,1964,2016-03-10,2016-12-19,157,Returning,6,52
6,00000,2016-12-19,1964,2016-08-14,2017-04-20,127,Returning,7,52
7,00000,2017-04-20,1964,2016-12-19,2018-03-19,122,Returning,8,53
8,00000,2018-03-19,1964,2017-04-20,2018-12-31,333,Returning,9,54
9,00000,2018-12-31,1964,2018-03-19,2020-09-06,287,Returning,10,54


In [36]:
con.close()